# Hard-weekend signal: an illustrative walkthrough

This notebook demonstrates the public-facing signal construction used by the project. U.S. equities are closed over the weekend, while prediction markets can continue to update. For an economically linked event--firm pair, the question is whether a signed prediction-market revision during that interval is reflected when the stock reopens.

> **Important:** Every number in this notebook is synthetic and generated below. The notebook is a software and research-design demonstration, not an empirical result, source-data release, or trading recommendation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the repository root importable whether Jupyter starts in the root
# folder or in the notebooks folder.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError('Open this notebook from inside the repository folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import build_weekend_signal


## 1. Create an illustrative event--firm panel

The real project obtains and validates this structure from multiple market-data sources. Here we generate a small deterministic example with the same public schema. `economic_sign` is +1 when a higher event probability is favorable to the linked firm and -1 when it is unfavorable.

In [ ]:
rng = np.random.default_rng(7)
n_observations = 48

friday_probability = rng.uniform(0.15, 0.85, n_observations)
probability_change = rng.normal(0, 0.09, n_observations)
sunday_probability = np.clip(friday_probability + probability_change, 0.01, 0.99)
economic_sign = rng.choice([-1, 1], size=n_observations)

# The relation below is deliberately simulated only to make the chart readable.
synthetic_signed_move_pp = 100 * economic_sign * (sunday_probability - friday_probability)
reopening_return_bps = 1.1 * synthetic_signed_move_pp + rng.normal(0, 8, n_observations)

illustrative_panel = pd.DataFrame(
    {
        'platform': np.where(np.arange(n_observations) % 2 == 0, 'example_a', 'example_b'),
        'market_key': [f'illustrative-event-{i % 12:02d}' for i in range(n_observations)],
        'stock_ticker': [f'DEMO{i % 6:02d}' for i in range(n_observations)],
        'session_date': pd.date_range('2026-01-05', periods=n_observations, freq='7D'),
        'friday_probability': friday_probability,
        'sunday_probability': sunday_probability,
        'economic_sign': economic_sign,
        'reopening_return_bps': reopening_return_bps,
    }
)

illustrative_panel.head()


## 2. Put heterogeneous links on one economic scale

A raw probability increase does not have the same interpretation for every firm. Multiplying by the reviewed economic sign makes a positive `signed_revision_pp` mean a favorable update for the linked firm across the panel.

In [ ]:
signals = build_weekend_signal(illustrative_panel)
signals[
    [
        'market_key',
        'stock_ticker',
        'friday_probability',
        'sunday_probability',
        'economic_sign',
        'probability_change_pp',
        'signed_revision_pp',
        'reopening_return_bps',
    ]
].head()


## 3. Visualize the illustrative relationship

The fitted line is intentionally based on the synthetic panel. In the real analysis, inference uses the protected final panel and additional design elements such as peer adjustments, fixed effects, diagnostics, and randomization checks.

In [ ]:
x = signals['signed_revision_pp'].to_numpy()
y = signals['reopening_return_bps'].to_numpy()
slope, intercept = np.polyfit(x, y, deg=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, color='#2A6F97', alpha=0.8, label='Synthetic event--firm observations')
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept + slope * x_line, color='#E76F51', linewidth=2, label='Synthetic fitted line')
ax.axhline(0, color='0.5', linewidth=0.8)
ax.axvline(0, color='0.5', linewidth=0.8)
ax.set(
    title='Illustrative hard-weekend signal construction',
    xlabel='Signed prediction-market revision (percentage points)',
    ylabel='Synthetic reopening return (basis points)',
)
ax.legend(frameon=False)
plt.show()

print(f'Synthetic fitted slope: {slope:.2f} basis points per signed probability point')


## What this public notebook does—and does not—show

It shows the transparent transformation from a before/after prediction-market probability to an economically signed event signal. It does **not** reproduce a paper table, disclose the reviewed mapping, or identify a causal effect of prediction markets on stocks. The full research workflow applies its timing, validation, peer-control, and inference procedures to protected inputs.